# Week 8 — End-to-End Business Analysis: Presentation Preparation
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

Wednesday you built the analysis. Today you turn it into something a leadership team will sit
through for ten minutes — and, just as importantly, you make every number on your slides
defensible.

By the end of this session, you will be able to:
- Prepare and deliver a **10-minute SQL-driven business analysis presentation** that synthesises the
  full course's SQL skills — JOINs, CTEs, window functions, `CASE WHEN` and date functions.
- Assemble the five required deliverables in SQL only, each one a query you can re-run live.
- Run a **final verification pass** over your figures before you present, so that no number on a
  slide is one you cannot reproduce on demand.

Run the setup cell first, as always. It rebuilds the SQLite database from the Olist CSVs and
connects the `%%sql` magic to it. Nothing below will run until this cell has finished.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters — the ten minutes you actually get

A presentation is not the analysis. The analysis is what you did Wednesday; the presentation is the
small part of it that survives contact with an audience who will not read SQL, will not see your
CTEs, and will remember perhaps five numbers.

Ten minutes across five deliverables is **two minutes each** — roughly one slide, one query, one
sentence of meaning per deliverable. That constraint is the whole exercise. You are not being asked
to show everything you found; you are being asked to decide what matters and to be right about it.

And being right is checkable. The platform holds **99,441 orders**, of which **96,478** were
delivered and **625** cancelled — **97.0%** delivered. Across every line item ever sold,
**R$15,843,553.24** of goods and freight changed hands. Those figures are settled. If a number on
your slide contradicts one of them, the contradiction is yours to explain, and the marking rubric —
SQL correctness 15, business insight 15, readability 10 — gives you no credit for a confident
delivery of a wrong figure. This notebook walks the five deliverables in order, then shows you the
verification pass to run immediately before you present.

## Group Exercise — finalise your presentation, five deliverables

⏱ the rest of this session · in your project groups

This is the assessed piece of work. By the end of today your group needs a finished 10-minute
presentation built on **five** deliverables, produced in SQL only (no pandas beyond displaying a
result):

1. **Platform overview KPIs** — total orders, total revenue, average order value, delivered %.
   This is your opening slide and your credibility: every figure here is independently verifiable.
2. **Top 5 product categories by revenue** — which categories actually drive the money, and what
   share of the platform each one carries.
3. **Seller tier breakdown** — reuse the CTE pattern you built in **Week 7**: total each seller
   first, then band the sellers into tiers and report the shape of the marketplace.
4. **Delivery performance** — average delivery days, late percentage, and how review scores move
   with delivery. Wednesday's Analysis 2 and Analysis 3 already did this work; today you compress
   it into one slide.
5. **One additional business question, chosen by your group** — the part that is yours. This is
   where "business insight quality" is won or lost.

How to split the next stretch of the session:

- **Assign an owner per deliverable**, exactly as you split the brief on Wednesday. Each owner is
  responsible for the query *and* for the one sentence that goes under it.
- **Agree the story before the slides.** Five disconnected queries score badly even when every
  number is right. Decide what your group's overall message is — the platform grew, but revenue
  concentrates in a handful of sellers and categories, and late delivery is what erodes
  satisfaction — and let each deliverable earn its place in that message.
- **Rehearse against the clock.** Two minutes per deliverable is less than it sounds.
- **Verify last, before anything else is polished.** The final section of this notebook shows you
  the two verification queries to run over your finished figures.

Each deliverable is demonstrated below. Read the query, understand the pattern, then write your
group's version — do not copy blindly, because your fifth deliverable will only work if you can
adapt the shapes you see here.

## Deliverable 1 — Platform overview KPIs

Your opening slide has one job: establish the size and health of the thing you are about to talk
about, in four numbers, in about thirty seconds. Orders placed, how many arrived, how many were
cancelled, and what proportion of the whole that represents.

The query below needs only Week 1, Week 2 and Week 4 material, which is the point — an opening
slide should be the *least* clever query in your deck, because it is the one everything else rests
on. `COUNT(DISTINCT order_id)` counts orders rather than rows. `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`
is the standard way to count "rows meeting a condition" alongside other aggregates in a single pass,
instead of running three separate queries and copying numbers between them.

One detail to notice and to repeat in your own queries: the delivered percentage uses
`THEN 1.0 ELSE 0`, not `THEN 1 ELSE 0`. Written with integers, SQLite divides one integer by another
and truncates the result to `0` — a silently wrong `0.0%` on a leadership slide.

In [ ]:
%%sql
-- Deliverable 1 — platform overview KPIs. Every value below is verified.
SELECT
    COUNT(DISTINCT order_id) AS total_orders,                                   -- Expected: 99,441
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered,   -- Expected: 96,478
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled,     -- Expected: 625
    ROUND(SUM(CASE WHEN order_status = 'delivered' THEN 1.0 ELSE 0 END)
          / COUNT(*) * 100, 1) AS delivered_pct                                 -- Expected: 97.0
FROM orders

### Reading the result

Four numbers, and a sentence each is plenty: *the platform handled 99,441 orders, delivered 97.0% of
them, and cancelled 625.*

The remaining slot in the brief's KPI list — **total revenue and average order value** — deliberately
does not live in this query, because `orders` holds no money. Revenue lives in `order_items` (price
and freight per line item) and in `order_payments` (what customers actually paid). Those two answer
subtly different questions and do **not** produce the same total. Pick one, say which one you picked,
and use it consistently across all five deliverables. The GMV figure from `order_items` —
**R$15,843,553.24** — is the one this notebook uses, and the verification section at the end
reproduces it.

> **Presentation note.** Do not put all of `total_orders`, `delivered`, `canceled` and `delivered_pct`
> on screen as a table of four. Lead with the two that carry the message — 99,441 orders, 97.0%
> delivered — and keep the rest in reserve for questions.

## Deliverable 2 — Top 5 product categories by revenue

"Which categories drive revenue?" needs three tables to answer, because Olist stores the answer in
pieces. The money is in `order_items` (`price` per line item). The category of the product sold is in
`products`, reachable by `product_id`. And that category is stored in Portuguese, so
`product_category_translation` turns it into the English name your audience can read.

The `WITH` block totals every category first; the outer `SELECT` then takes the top five and adds
each one's share of categorised revenue using a scalar sub-query over the CTE. Two things are worth
copying into your own work:

- **`COUNT(DISTINCT oi.order_id)`, not `COUNT(*)`.** `order_items` is at line-item grain — 112,650
  rows across 98,666 orders — so `COUNT(*)` would count items and quietly overstate how many
  customers bought in each category.
- **This query joins only *one* of the many-per-order tables.** `order_items` is here; `order_payments`
  and `order_reviews` are not. Adding either without collapsing it first would multiply rows and
  inflate every revenue figure. The Common Mistakes section shows exactly what that damage looks like.

In [ ]:
%%sql
-- Deliverable 2 — the five categories that actually carry the revenue.
WITH category_revenue AS (
    SELECT t.product_category_name_english      AS category,
           SUM(oi.price)                        AS product_revenue,
           COUNT(DISTINCT oi.order_id)          AS order_count
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_translation t
         ON p.product_category_name = t.product_category_name
    GROUP BY t.product_category_name_english
)
SELECT category,
       ROUND(product_revenue, 2) AS product_revenue,   -- health_beauty Expected: 1,258,681.34
       order_count,
       ROUND(product_revenue * 100.0
             / (SELECT SUM(product_revenue) FROM category_revenue), 1) AS pct_of_categorised_revenue
FROM category_revenue
ORDER BY product_revenue DESC
LIMIT 5

### Reading the result

The top row is your verification anchor: **health_beauty** at **R$1,258,681.34**. If your group's
version of this query produces anything else for health_beauty, something in your joins is wrong —
almost always an extra many-per-order table — and you should fix it before the number reaches a slide.

Then read the `pct_of_categorised_revenue` column, because that is the insight rather than the fact.
No single category dominates this marketplace: the leader carries well under a tenth of categorised
revenue, and the top five together carry a minority of it. That is a genuine finding with a business
consequence — a marketplace whose revenue is spread thin across many categories cannot be grown by
betting on one of them, which is a different strategy conversation from the one leadership might
expect.

Note also that `order_count` does not rank in the same order as revenue: a category can sell to fewer
customers and still out-earn one that sells to more, because the basket is worth more. Saying that
out loud — *these are high-value categories, not high-volume ones* — is the kind of remark that
earns business-insight marks.

## Deliverable 3 — Seller tier breakdown

This deliverable reuses the Week 7 pattern directly: **a CTE that collapses the data to the grain you
actually want to reason about, and a second step that classifies it.** Here the grain you want is one
row per *seller*, not per line item — so `seller_revenue` totals each seller's sales first, and every
later step works on those 3,095 rows rather than on 112,650 item rows.

The second CTE tiers the sellers with `NTILE(4) OVER (ORDER BY revenue DESC)` — the window function
from Week 5 that splits an ordered result into four equal-sized buckets. `ORDER BY revenue DESC` puts
the biggest sellers in tier 1. `NTILE` is the right tool when you want *equal-sized groups*; if your
group would rather band sellers by fixed thresholds ("above R$100,000", "R$10,000–100,000", …), use
`CASE WHEN` over the same `seller_revenue` CTE instead. Both are defensible — but say which you chose
and why, because equal-sized quartiles and fixed money bands tell different stories.

The final `SELECT` aggregates the tiers: how many sellers are in each, how much revenue the tier
carries, the average seller in it, and the largest single seller it contains.

In [ ]:
%%sql
-- Deliverable 3 — Week 7's CTE pattern: collapse to seller grain, then tier.
WITH seller_revenue AS (
    SELECT seller_id,
           SUM(price)                 AS revenue,
           COUNT(DISTINCT order_id)   AS orders
    FROM order_items
    GROUP BY seller_id
),
tiered_sellers AS (
    SELECT seller_id,
           revenue,
           orders,
           NTILE(4) OVER (ORDER BY revenue DESC) AS tier
    FROM seller_revenue
)
SELECT tier,
       COUNT(*)                        AS sellers,
       ROUND(SUM(revenue), 2)          AS tier_revenue,
       ROUND(AVG(revenue), 2)          AS avg_seller_revenue,
       ROUND(MAX(revenue), 2)          AS top_seller_in_tier,   -- tier 1 Expected: 229,472.63
       ROUND(SUM(revenue) * 100.0
             / (SELECT SUM(revenue) FROM seller_revenue), 1) AS pct_of_seller_revenue
FROM tiered_sellers
GROUP BY tier
ORDER BY tier

### Reading the result

Start with the verification: `top_seller_in_tier` for tier 1 should read **R$229,472.63** — the
platform's largest seller by product revenue, and a figure you can check against the course's verified
stats. It appears here for free, because the top of tier 1 *is* the top of the marketplace.

Then read the `pct_of_seller_revenue` column down the four rows, and notice how steeply it falls. The
top quarter of sellers carries the overwhelming majority of revenue; the bottom quarter carries almost
none. That is the classic marketplace concentration curve, and it is the single most presentable thing
in this deliverable — far more interesting than the tier counts, which are equal by construction
(`NTILE(4)` makes them equal, so *do not* present "there are roughly 774 sellers per tier" as a
finding; it is an artefact of your own method).

The business reading is where your marks are: a platform this concentrated has a small number of
sellers it cannot afford to lose, and a long tail whose economics are worth questioning. Pair that
with the previous deliverable — revenue spread thinly across categories, but concentrated tightly
among sellers — and you have a genuine insight rather than two tables.

## Deliverable 4 — Delivery performance and satisfaction, on one slide

You already built this on Wednesday. **Analysis 2** broke delivery down by customer state, and
**Analysis 3** broke it down by review score — both worth having in your back pocket for questions.
Neither fits a two-minute slot: one returns a state-by-state table, the other five rows that need
narrating. For the presentation you want the headline version.

The query below compresses both into a single row, and it demonstrates the grain discipline that has
run through the whole course. `order_reviews` holds **many rows per order** (99,224 reviews across
98,673 orders), so the `rev` CTE collapses it to exactly one row per order *before* it is joined. The
`del` CTE does the parallel job for delivery: one row per delivered order, carrying the duration in
days and a late flag. Only then do the two meet — 1:1, no fan-out, safe to aggregate.

Three Week-4 ideas do the work: `julianday(...) - julianday(...)` measures a gap between two TEXT
timestamps (SQLite has no date type and no `DATEDIFF`), `CASE WHEN delivered > estimated THEN 1 ELSE 0`
defines *late*, and `* 100.0` forces the percentage into REAL arithmetic. The two `AVG(CASE WHEN ...)`
columns at the end are the payoff: the average review score for late orders and for on-time ones, side
by side.

In [ ]:
%%sql
-- Deliverable 4 — delivery headline + the review-score gap, fan-out safe.
WITH rev AS (                       -- collapse reviews to ONE row per order first
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
),
del AS (                            -- one row per delivered order: duration + late flag
    SELECT o.order_id,
           julianday(o.order_delivered_customer_date)
             - julianday(o.order_purchase_timestamp) AS delivery_days,
           CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1 ELSE 0 END AS is_late
    FROM orders o
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
)
SELECT COUNT(*)                                   AS delivered_orders,
       ROUND(AVG(d.delivery_days), 1)             AS avg_delivery_days,   -- Expected: 12.6
       SUM(d.is_late)                             AS late_orders,         -- Expected: 7,826
       ROUND(SUM(d.is_late) * 100.0 / COUNT(*), 1) AS late_pct,           -- Expected: 8.1
       ROUND(AVG(CASE WHEN d.is_late = 1 THEN r.review_score END), 2) AS avg_score_late,
       ROUND(AVG(CASE WHEN d.is_late = 0 THEN r.review_score END), 2) AS avg_score_on_time
FROM del d
LEFT JOIN rev r ON d.order_id = r.order_id

### Reading the result

Three of these values are verified and should match exactly: **12.6** average delivery days,
**7,826** late orders, **8.1%** late. `delivered_orders` comes in a shade under the 96,478 orders with
status `delivered`, because a small number of them carry no delivery timestamp and the `IS NOT NULL`
filter removes them — a difference worth knowing about before somebody asks.

The last two columns are your slide. The average review score for late orders sits far below the score
for on-time ones — a gap of well over a point and a half, against an overall platform average of
**4.09**. One sentence: *when we deliver on time customers rate us around four and a half stars; when
we are late they rate us under three.*

Two cautions to state before your audience finds them. This is **association, not proof of cause** —
late orders may also be the far-away, bulky or expensive ones. And every row here is a **delivered**
order: customers whose orders never arrived are excluded entirely, and they are unlikely to be the
happy ones. Naming a limitation yourself is what "business insight quality" means on the rubric.

## Deliverable 5 — your group's own business question

The fifth deliverable has no worked example here, and that is deliberate. Four questions were handed
to you; this one measures whether you can find one. A good capstone question has three properties:

1. **This dataset can actually answer it.** Olist gives you orders, items, payments, reviews,
   products, sellers and customer state — and nothing about marketing spend, stock levels, margins or
   customer demographics. A question requiring those is a good question and the wrong one for today.
2. **The answer could plausibly change a decision.** "How many orders were placed?" is a fact.
   "Which categories have the worst late-delivery rate, and are they also the worst-rated?" is a
   question whose answer tells somebody what to do on Monday.
3. **You can verify it.** You must be able to tie some part of the result back to a figure you already
   trust — a total that reconciles, a count that matches, an anchor you can quote.

Starting points, though a question of your own scores better than one borrowed from this list:

- Which payment type carries the highest average order value, and has that shifted between 2017 and 2018?
- How concentrated is the platform really — what share of revenue comes from the top 10 sellers alone?
- Which categories combine high revenue with poor review scores? (Use the `rev` CTE pattern from
  Deliverable 4 — never join raw `order_reviews` to `order_items`.)
- Which month was the platform's best, and can you explain the spike? November 2017 carries **7,544**
  orders, and there is a reason.
- Do sellers in tier 1 deliver faster than sellers in tier 4, and does that show up in their reviews?

Rules that carry marks: **SQL only**, **use a CTE** when the query needs more than one step, **watch
the grain** if your question touches two of `order_items` / `order_payments` / `order_reviews`, and
**sanity-check the number** before it reaches a slide.

Replace the placeholder below with your group's query. Sketch the steps as comments first, then build
the CTE, then the outer `SELECT`.

In [ ]:
%%sql
-- Deliverable 5 — your group's chosen question. Replace this placeholder.
-- Step 1 (comment your plan first):  what grain do I need — order? seller? category? month?
-- Step 2:  WITH ... AS ( ... )       collapse to that grain
-- Step 3:  SELECT ... FROM ...       present it, with a share or a benchmark column
SELECT 'write your group question here' AS todo

---
## 🤖 Using DeepSeek on presentation day

Today DeepSeek is most useful for the two jobs that are not query-writing: **tightening the sentence
under each number**, and **stress-testing your interpretation** before an audience does it for you.

For query help, the prompt-then-verify protocol from Week 4 still applies without exception, and on
presentation day it tightens:

1. **Prompt with the schema and the grain.** Name the tables and columns, and say which tables hold
   many rows per order. Without that, DeepSeek will cheerfully join `order_items` to `order_reviews`
   and hand you an inflated total.
2. **Read the SQL before you run it.** Three checks: does it join two many-per-order tables? does it
   divide integers without `* 1.0`? does it use `= NULL`? Those are the errors this dataset provokes.
3. **Make it reproduce a number you already trust** — 99,441 orders, R$15,843,553.24 GMV,
   health_beauty at R$1,258,681.34. A query that cannot reproduce a figure you know cannot be trusted
   with one you don't.

A useful non-SQL prompt for today: *"Here is my finding and the query behind it. What would a sceptical
executive ask me about this?"* Then prepare those answers. What you must never do is let a number reach
a slide because an AI said it — the verification section below is how you prevent that.

## Going deeper — the verification pass, run this last

This is the ritual to perform immediately before you present: re-run the figures your deck rests on,
and confirm they still match. It takes thirty seconds and it is the difference between a presentation
you can defend and one you can only hope survives.

The query below is the revenue half of that pass — total product revenue, total freight revenue and
total GMV, straight from `order_items` at line-item grain, with no joins at all to go wrong. The KPI
half is Deliverable 1's query, which you have already run above. Between them, those two queries
anchor every money figure and every order-count figure in your deck.

There is a second, subtler check hiding here, and it is a good slide in its own right: the revenue
reported by Deliverable 2 sums the **categorised** items only. Roughly six hundred products carry no
category at all, so their sales belong to no category and the category table cannot add up to the
platform total. The second query below quantifies that gap. Knowing it is there — and saying so —
protects you from the one question guaranteed to be asked: *"why don't your category numbers add up
to your total?"*

In [ ]:
%%sql
-- Verification pass, part 1 of 2 — the money anchors. No joins, nothing to fan out.
SELECT ROUND(SUM(price), 2)                 AS product_revenue,   -- Expected: 13,591,643.70
       ROUND(SUM(freight_value), 2)         AS freight_revenue,   -- Expected: 2,251,909.54
       ROUND(SUM(price + freight_value), 2) AS total_gmv          -- Expected: 15,843,553.24
FROM order_items

In [ ]:
%%sql
-- Verification pass, part 2 of 2 — why category revenue is LESS than platform revenue.
WITH category_revenue AS (
    SELECT t.product_category_name_english AS category,
           SUM(oi.price)                   AS product_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_translation t
         ON p.product_category_name = t.product_category_name
    GROUP BY t.product_category_name_english
)
SELECT COUNT(*)                                          AS categories,
       ROUND(SUM(product_revenue), 2)                    AS categorised_revenue,
       (SELECT ROUND(SUM(price), 2) FROM order_items)    AS all_item_revenue,  -- Expected: 13,591,643.70
       ROUND((SELECT SUM(price) FROM order_items)
             - SUM(product_revenue), 2)                  AS uncategorised_gap
FROM category_revenue

### Reading the result

The first query must reproduce **R$13,591,643.70**, **R$2,251,909.54** and **R$15,843,553.24** exactly.
If it does, every money figure in your deck that derives from `order_items` is standing on solid ground.

The second query explains the gap: `categorised_revenue` is visibly lower than `all_item_revenue`, and
`uncategorised_gap` is the difference — sales of products whose `product_category_name` is missing, so
the join to `products` and the translation table finds no category for them. That is an inner join
quietly dropping rows, which is exactly the behaviour you were warned about in Week 3, appearing here
in a place where it is *correct* but still needs disclosing.

> **On the slide, this is one line:** *"category figures cover categorised products only, which is
> about 98–99% of revenue."* Said in advance, it is rigour. Discovered by your audience, it is a hole.

## Common mistakes — the ones that reach slides

**Mistake 1 — join fan-out.** `order_items`, `order_payments` and `order_reviews` all hold many rows
per order. Join two of them and then `SUM` money, and SQLite hands you every combination: an order with
3 items and 2 payments becomes 6 rows. Nothing errors, the number looks plausible, and it is wrong.
This is *the* capstone killer, and it is why Deliverable 4 collapses `order_reviews` to one row per
order in a CTE before joining. The cell below shows the wrong version as a comment — never run a query
that produces a wrong number you might accidentally copy — then runs the correct one.

**Mistake 2 — mixing revenue definitions between slides.** `SUM(order_items.price + freight_value)`
and `SUM(order_payments.payment_value)` are both legitimate and they are **not equal**. Using one on
slide 2 and the other on slide 4 guarantees a question you cannot answer. Choose once, state the
choice, keep it.

**Mistake 3 — integer division.** `late_count * 100 / delivered_orders` returns `0`, every time,
because SQLite divides integers into an integer. Write `* 100.0`.

**Mistake 4 — presenting an artefact as a finding.** "Each tier has about 774 sellers" is true and
meaningless: `NTILE(4)` made it so. Present what the data decided, not what your method decided.

In [ ]:
%%sql
-- ── COMMON MISTAKE — join fan-out ───────────────────────────────────
-- WRONG — joins TWO many-per-order tables, then averages a review score:
--   SELECT ROUND(AVG(r.review_score), 2) AS avg_score
--   FROM order_items oi
--   JOIN order_reviews r ON oi.order_id = r.order_id
--   -> each review is repeated once per line item, so multi-item orders are
--      weighted more heavily. No error, no warning, wrong number.
--
-- CORRECT — collapse reviews to one row per order FIRST, then join 1:1.
WITH rev AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
),
order_items_per_order AS (
    SELECT order_id, SUM(price) AS order_product_revenue
    FROM order_items
    GROUP BY order_id
)
SELECT COUNT(*)                            AS orders_with_items_and_review,
       ROUND(AVG(r.review_score), 2)       AS avg_score_per_order,
       ROUND(SUM(i.order_product_revenue), 2) AS product_revenue_of_those_orders
FROM order_items_per_order i
JOIN rev r ON i.order_id = r.order_id

## Mini-challenge — your headline number

⏱ ~5 min · individually, before you rejoin your group

Every good presentation has one number the audience remembers. Not five — one. Your group's deck has
one too, whether you chose it or not.

Individually, decide what yours should be, and write the single query that produces it. Constraints:

- **One query, one row, at most three columns.** If it needs more than that, it is not a headline.
- **It must be verifiable** — either it reproduces an anchor you already trust, or it is built from
  one in a way you can explain in a sentence.
- **Write the sentence too**, as a comment above your query. If you cannot say what the number means
  in one line, it is the wrong number.

Then compare with the rest of your group. If three of you picked different headlines, your story is
not yet agreed — and that conversation is more valuable than another query.

Replace the placeholder below.

In [ ]:
%%sql
-- ⏱ ~5 min — your turn! Replace the placeholder with your headline query.
-- The sentence it supports (write this FIRST, in one line):
--   "..."
SELECT 'write your headline query here' AS todo

## Session Summary

Five deliverables, one story, ten minutes. What each one needs and where the SQL came from:

| Deliverable | The query shape | Skills it uses |
|---|---|---|
| 1 — Platform overview KPIs | single-row aggregate over `orders` | `COUNT(DISTINCT ...)`, `SUM(CASE WHEN ...)`, REAL division (W1, W2, W4) |
| 2 — Top 5 categories by revenue | CTE totalling categories, then `ORDER BY ... LIMIT 5` | three-table JOIN, `GROUP BY`, share-of-total sub-query (W3, W6, W7) |
| 3 — Seller tier breakdown | Week 7's CTE to seller grain, then `NTILE(4)` | CTE chaining, window function, tier aggregation (W5, W7) |
| 4 — Delivery performance | two CTEs collapsed to one row per order, joined 1:1 | `julianday`, `CASE WHEN`, fan-out discipline (W4, W6) |
| 5 — Your own question | yours to design | all of it |

Three things to carry onto the stage:

1. **Every number is verifiable, and you verify them last.** 99,441 orders · 96,478 delivered · 97.0%
   delivered · R$15,843,553.24 GMV · health_beauty R$1,258,681.34 · top seller R$229,472.63 ·
   12.6 average delivery days · 8.1% late. Re-run the verification pass before you present.
2. **Grain before aggregation.** Collapse many-per-order tables to one row per order in a CTE before
   you `SUM` or `AVG`. Almost every wrong capstone number traces back to this.
3. **A number without a sentence is not a finding.** The rubric gives 15 marks to SQL correctness and
   15 to business insight — the SQL only buys you the right to be listened to.

---
**Coming up**: this is the final session of Phase 2b. Your group presentation is assessed on SQL
correctness (15), business insight quality (15) and query readability and structure (10) — 40 marks in
total. Next comes **Phase 2c**, the capstone project, where Python and SQL meet: the queries you wrote
here get wrapped in pandas and shipped as a Streamlit app.